# Aula 03 · Treinamento de uma MLP

Na aprovação de um pedido, já há sinais de atraso? Vamos usar três atributos da Olist para treinar a mesma rede em NumPy, TensorFlow/Keras e PyTorch.

Execute as seções em ordem com o kernel da pasta `codigos`. TensorFlow e PyTorch usam GPU quando disponível. Os notebooks da aula e da atividade usam os CSVs em `codigos/dados/brutos/`.

## 01 · Dados, dimensões e arquitetura

Cada linha de `pedidos` representa um pedido. Em `itens`, cada linha representa um item. Somamos os valores e os fretes e contamos os itens por `order_id` para obter três atributos por pedido.

In [ ]:
from os.path import isdir

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

raiz = "." if isdir("aula03") else ".."
pasta = f"{raiz}/dados/brutos"

datas = [
    "order_approved_at",
    "order_estimated_delivery_date",
    "order_delivered_customer_date",
]
pedidos = pd.read_csv(f"{pasta}/olist_orders_dataset.csv", parse_dates=datas)
itens = pd.read_csv(f"{pasta}/olist_order_items_dataset.csv")

pedidos[["order_id", "order_status", *datas]].head()

In [ ]:
itens[["order_id", "order_item_id", "price", "freight_value"]].head()

### Uma linha por pedido

Usamos pedidos entregues e com as datas necessárias. O alvo vale 1 quando o **dia** da entrega real é posterior ao prometido. Na aprovação, as entradas são o valor dos itens, o frete e a quantidade.

In [ ]:
itens_por_pedido = itens.groupby("order_id", as_index=False).agg(
    valor_itens=("price", "sum"),
    frete=("freight_value", "sum"),
    quantidade_itens=("order_item_id", "count"),
)
entregues = pedidos.loc[pedidos["order_status"] == "delivered"].dropna(subset=datas)
dados = entregues.merge(itens_por_pedido, on="order_id", validate="one_to_one")

aprovacao = dados["order_approved_at"]
entrega = dados["order_delivered_customer_date"]
dados = dados.loc[entrega >= aprovacao].copy()
dia_real = dados["order_delivered_customer_date"].dt.normalize()
dia_prometido = dados["order_estimated_delivery_date"].dt.normalize()
dados["alvo"] = (dia_real > dia_prometido).astype(int)

atributos = ["valor_itens", "frete", "quantidade_itens"]
dados[["order_approved_at", *atributos, "alvo"]].head()

### Treino, validação e teste

A unidade da divisão é o pedido. Usamos todos os pedidos elegíveis e separamos **60% para treino, 20% para validação e 20% para teste**. A estratificação mantém proporções semelhantes de atrasos.

Ajustamos a padronização no treino e aplicamos os mesmos valores à validação. O teste será usado na atividade, após escolher o modelo pela validação.

In [ ]:
SEMENTE = 42
OCULTOS = 16
EPOCAS = 30
LOTE = 48
TAXA = 0.05

In [ ]:
dados = dados.sort_values(["order_approved_at", "order_id"])

treino, restante = train_test_split(
    dados, train_size=0.6, stratify=dados["alvo"], random_state=SEMENTE
)
validacao, teste = train_test_split(
    restante, train_size=0.5, stratify=restante["alvo"], random_state=SEMENTE
)

In [ ]:
escala = StandardScaler()
X = escala.fit_transform(treino[atributos])
Xv = escala.transform(validacao[atributos])
y = treino[["alvo"]].to_numpy(dtype=np.float64)
yv = validacao[["alvo"]].to_numpy(dtype=np.float64)

print(f"Treino {X.shape} · {int(y.sum())} atrasos")
print(f"Validação {Xv.shape} · {int(yv.sum())} atrasos")
print(f"Teste · {len(teste)} pedidos reservados")

### Um pedido real

Veja as três entradas do mesmo pedido antes e depois da padronização. O alvo é obtido comparando o dia da entrega real com o prometido.

In [ ]:
exemplo = dados.loc[dados["order_id"] == "f4bc8c5c2d0700575efaf3a5f913bc65"]
exemplo[["order_id", *datas, *atributos, "alvo"]]

In [ ]:
pd.DataFrame(
    escala.transform(exemplo[atributos]), columns=atributos
).round(4)

### A rede do exemplo

A MLP tem **3 entradas**, **16 unidades tanh** e **1 logit por pedido**. A sigmoide transforma o logit na probabilidade estimada de atraso. Usaremos os mesmos pesos iniciais e a mesma ordem de lotes nas três implementações.

In [ ]:
rng = np.random.default_rng(SEMENTE)
entradas = X.shape[1]
inicial = {
    "W1": rng.normal(size=(entradas, OCULTOS)) / np.sqrt(entradas),
    "b1": np.zeros(OCULTOS),
    "W2": rng.normal(size=(OCULTOS, 1)) / np.sqrt(OCULTOS),
    "b2": np.zeros(1),
}

def lotes_de_treino(total, rng):
    ordem = rng.permutation(total)
    for inicio in range(0, total, LOTE):
        yield ordem[inicio : inicio + LOTE]

print("Parâmetros treináveis", sum(valor.size for valor in inicial.values()))

## 02 · Ciclo de treinamento em NumPy

Em cada lote, calculamos as ativações, os logits e os gradientes. Para um pedido, o sinal no logit é `p − y`, com `p = sigmoid(z)`. Ao derivar a perda média, dividimos esse sinal por `B`, o tamanho do lote. Depois de obter todos os gradientes, aplicamos um passo de SGD.

Ao fim da época, medimos a perda no treino e na validação com os mesmos pesos.

In [ ]:
def sigmoide(z):
    return np.exp(-np.logaddexp(0.0, -z))

def propagacao_numpy(X, parametros):
    H = np.tanh(X @ parametros["W1"] + parametros["b1"])
    z = H @ parametros["W2"] + parametros["b2"]
    return H, z

In [ ]:
def perda_numpy(y, z):
    return np.mean(np.logaddexp(0.0, z) - y * z)

In [ ]:
def gradientes_numpy(X, y, parametros):
    H, z = propagacao_numpy(X, parametros)
    dz = (sigmoide(z) - y) / len(X)
    dH = (dz @ parametros["W2"].T) * (1 - H**2)
    return {
        "W1": X.T @ dH,
        "b1": dH.sum(axis=0),
        "W2": H.T @ dz,
        "b2": dz.sum(axis=0),
    }

### Do gradiente de um pedido ao gradiente do lote

Vamos acompanhar um peso de saída, `v₁`. A contribuição de um pedido é `h₁(p − y)`. O gradiente da perda média é a média dessas contribuições, calculadas com os mesmos pesos.

In [ ]:
primeiro_lote = next(lotes_de_treino(len(X), np.random.default_rng(SEMENTE)))
H, z = propagacao_numpy(X[primeiro_lote], inicial)
p = sigmoide(z).ravel()
alvos = y[primeiro_lote].ravel()
contribuicoes = H[:, 0] * (p - alvos)
pedido_ilustrado = 2

print(f"Pedido do lote {pedido_ilustrado + 1} · h₁ {H[pedido_ilustrado, 0]:.5f} · p {p[pedido_ilustrado]:.5f} · y {alvos[pedido_ilustrado]:.0f}")
print(f"Gradiente desse pedido para v₁ {contribuicoes[pedido_ilustrado]:.5f}")
soma_outros = contribuicoes.sum() - contribuicoes[pedido_ilustrado]
print(f"Soma das contribuições dos outros {len(contribuicoes) - 1} pedidos {soma_outros:.5f}")
print(f"Gradiente médio do lote para v₁ {contribuicoes.mean():.5f}")

In [ ]:
parametros_numpy = {nome: valor.copy() for nome, valor in inicial.items()}
rng = np.random.default_rng(SEMENTE)
historico_numpy = []

for _ in range(EPOCAS):
    for ids in lotes_de_treino(len(X), rng):
        gradientes = gradientes_numpy(X[ids], y[ids], parametros_numpy)
        for nome in parametros_numpy:
            parametros_numpy[nome] -= TAXA * gradientes[nome]

    z_treino = propagacao_numpy(X, parametros_numpy)[1]
    z_validacao = propagacao_numpy(Xv, parametros_numpy)[1]
    historico_numpy.append((perda_numpy(y, z_treino), perda_numpy(yv, z_validacao)))

prob_numpy = sigmoide(z_validacao)
print("BCE final em NumPy", historico_numpy[-1])

## 03 · A mesma MLP em TensorFlow/Keras

Como na Aula 02, `GradientTape` registra a ida e calcula os gradientes da perda. As camadas `Dense` recebem os pesos do NumPy, e o otimizador SGD atualiza as variáveis treináveis.

In [ ]:
import tensorflow as tf

def criar_modelo_tf():
    return tf.keras.Sequential([
        tf.keras.Input(shape=(entradas,), dtype="float64"),
        tf.keras.layers.Dense(OCULTOS, activation="tanh", dtype="float64"),
        tf.keras.layers.Dense(1, dtype="float64"),
    ])

def perda_tf(y, z):
    return tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=y, logits=z))

In [ ]:
gpus_tf = tf.config.list_physical_devices("GPU")
for gpu in gpus_tf:
    tf.config.experimental.set_memory_growth(gpu, True)
dispositivo_tf = "/GPU:0" if gpus_tf else "/CPU:0"
print("Dispositivo TensorFlow", dispositivo_tf)

with tf.device(dispositivo_tf):
    modelo_tf = criar_modelo_tf()
    modelo_tf.set_weights([inicial[nome] for nome in ("W1", "b1", "W2", "b2")])
    sgd_tf = tf.keras.optimizers.SGD(learning_rate=TAXA)
    X_tf, y_tf, Xv_tf, yv_tf = [tf.convert_to_tensor(a) for a in (X, y, Xv, yv)]
    rng = np.random.default_rng(SEMENTE)
    historico_tf = []

    for _ in range(EPOCAS):
        for ids in lotes_de_treino(len(X), rng):
            xb, yb = tf.gather(X_tf, ids), tf.gather(y_tf, ids)
            with tf.GradientTape() as fita:
                perda_lote = perda_tf(yb, modelo_tf(xb, training=True))
            variaveis = modelo_tf.trainable_variables
            gradientes = fita.gradient(perda_lote, variaveis)
            sgd_tf.apply_gradients(zip(gradientes, variaveis))

        z_treino = modelo_tf(X_tf, training=False)
        z_validacao = modelo_tf(Xv_tf, training=False)
        historico_tf.append((
            float(perda_tf(y_tf, z_treino)),
            float(perda_tf(yv_tf, z_validacao)),
        ))

    prob_tf = tf.sigmoid(z_validacao).numpy()

print("BCE final em TensorFlow", historico_tf[-1])

## 04 · A mesma MLP em PyTorch

Nesta versão, `backward()` calcula os gradientes e `step()` atualiza os parâmetros. Limpamos os gradientes antes de cada lote. A camada `Linear` armazena os pesos na ordem saídas × entradas, portanto copiamos as matrizes transpostas.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)

def criar_modelo_torch():
    return nn.Sequential(nn.Linear(entradas, OCULTOS), nn.Tanh(), nn.Linear(OCULTOS, 1)).double()

In [ ]:
modelo_torch = criar_modelo_torch()
with torch.no_grad():
    modelo_torch[0].weight.copy_(torch.from_numpy(inicial["W1"].T))
    modelo_torch[0].bias.copy_(torch.from_numpy(inicial["b1"]))
    modelo_torch[2].weight.copy_(torch.from_numpy(inicial["W2"].T))
    modelo_torch[2].bias.copy_(torch.from_numpy(inicial["b2"]))

In [ ]:
dispositivo_torch = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
modelo_torch = modelo_torch.to(dispositivo_torch)
print("Dispositivo PyTorch", dispositivo_torch)

sgd_torch = torch.optim.SGD(modelo_torch.parameters(), lr=TAXA)
criterio = nn.BCEWithLogitsLoss()
X_torch, y_torch, Xv_torch, yv_torch = [
    torch.from_numpy(a).to(dispositivo_torch) for a in (X, y, Xv, yv)
]
rng = np.random.default_rng(SEMENTE)
historico_torch = []

for _ in range(EPOCAS):
    modelo_torch.train()
    for ids in lotes_de_treino(len(X), rng):
        sgd_torch.zero_grad()
        perda_lote = criterio(modelo_torch(X_torch[ids]), y_torch[ids])
        perda_lote.backward()
        sgd_torch.step()

    modelo_torch.eval()
    with torch.no_grad():
        z_treino = modelo_torch(X_torch)
        z_validacao = modelo_torch(Xv_torch)
        historico_torch.append((
            criterio(z_treino, y_torch).item(),
            criterio(z_validacao, yv_torch).item(),
        ))

prob_torch = torch.sigmoid(z_validacao).cpu().numpy()
print("BCE final em PyTorch", historico_torch[-1])

## 05 · Curvas de aprendizagem e métricas de validação

A entropia cruzada binária (BCE) mede a perda que minimizamos no treino. A Average Precision (AP) avalia a ordem dos pedidos pelas probabilidades estimadas de atraso. Comparamos as três MLPs com um escore constante e uma regressão logística ajustada nas mesmas três colunas.

Calculamos AP com as probabilidades. Para acurácia, precisão, recall e F1, classificamos como atraso os pedidos com probabilidade de pelo menos 0,5.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score

historicos = {
    "NumPy": historico_numpy,
    "TensorFlow": historico_tf,
    "PyTorch": historico_torch,
}
fig, eixos = plt.subplots(1, 3, figsize=(12, 3.4), sharey=True, layout="constrained")
for eixo, (nome, historico) in zip(eixos, historicos.items()):
    perdas = np.array(historico)
    epocas = np.arange(1, len(perdas) + 1)
    eixo.plot(epocas, perdas[:, 0], label="Treino")
    eixo.plot(epocas, perdas[:, 1], "--", label="Validação")
    eixo.set(title=nome, xlabel="Época")
    eixo.grid(alpha=0.2)
eixos[0].set_ylabel("BCE média")
eixos[-1].legend()
plt.show()

In [ ]:
logistica = LogisticRegression().fit(X, y.ravel())
escores = {
    "Dummy": np.full(len(yv), y.mean()),
    "Regressão logística": logistica.predict_proba(Xv)[:, 1],
    "MLP · NumPy": prob_numpy.ravel(),
    "MLP · TensorFlow": prob_tf.ravel(),
    "MLP · PyTorch": prob_torch.ravel(),
}
alvos_validacao = yv.ravel()
linhas = []
for nome, escore in escores.items():
    classes = (escore >= 0.5).astype(int)
    linhas.append({
        "Modelo": nome,
        "AP de validação": average_precision_score(alvos_validacao, escore),
        "Acurácia": accuracy_score(alvos_validacao, classes),
        "Precisão": precision_score(alvos_validacao, classes, zero_division=0),
        "Recall": recall_score(alvos_validacao, classes, zero_division=0),
        "F1": f1_score(alvos_validacao, classes, zero_division=0),
    })

resultados = pd.DataFrame(linhas)
resultados.round(4)

In [ ]:
for nome, prob in [("TensorFlow", prob_tf), ("PyTorch", prob_torch)]:
    diferenca = np.max(np.abs(prob_numpy - prob))
    print(f"Maior diferença de escore em relação ao NumPy · {nome} {diferenca:.2e}")

## 06 · Salvamento, recuperação e inferência

Salve os pesos de cada implementação e recupere a mesma arquitetura. Compare os escores para um lote de validação já padronizado. Para usar dados brutos em outra sessão, também é necessário reaplicar a padronização ajustada no treino.

In [ ]:
from os import makedirs

destino = f"{raiz}/resultados/aula03/referencia"
makedirs(destino, exist_ok=True)
caminho_numpy = f"{destino}/mlp_numpy.npz"
caminho_tf = f"{destino}/mlp_tensorflow.weights.h5"
caminho_torch = f"{destino}/mlp_pytorch.pt"

np.savez(caminho_numpy, **parametros_numpy)
modelo_tf.save_weights(caminho_tf)
torch.save(modelo_torch.state_dict(), caminho_torch)

In [ ]:
lote = Xv[:8]
with np.load(caminho_numpy) as pesos:
    recuperado_numpy = sigmoide(propagacao_numpy(lote, pesos)[1])

with tf.device("/CPU:0"):
    copia_tf = criar_modelo_tf()
    copia_tf.load_weights(caminho_tf)
    recuperado_tf = tf.sigmoid(copia_tf(lote, training=False)).numpy()

copia_torch = criar_modelo_torch()
copia_torch.load_state_dict(torch.load(caminho_torch, map_location="cpu", weights_only=True))
copia_torch.eval()
with torch.no_grad():
    recuperado_torch = torch.sigmoid(copia_torch(torch.from_numpy(lote))).numpy()

pd.DataFrame({
    "Modelo": ["NumPy", "TensorFlow", "PyTorch"],
    "Maior diferença após recuperar": [
        np.max(np.abs(prob_numpy[:8] - recuperado_numpy)),
        np.max(np.abs(prob_tf[:8] - recuperado_tf)),
        np.max(np.abs(prob_torch[:8] - recuperado_torch)),
    ],
})

## Referências

- [Olist · Brazilian E-Commerce Public Dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
- [TensorFlow · Diferenciação automática](https://www.tensorflow.org/guide/autodiff)
- [PyTorch · Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [scikit-learn · Average Precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.average_precision_score.html)